## 전문가 지식 근로자 (Expert Knowledge Worker)

### 전문 지식 근로자 역할을 하는 질문-답변 에이전트
### 사용 대상: Insurellm(보험 기술 회사)의 직원들
### 에이전트는 정확해야 하며, 솔루션은 저비용이어야 합니다.

이 프로젝트는 **RAG(검색 증강 생성, Retrieval Augmented Generation)**를 사용하여 질문-답변 어시스턴트의 높은 정확도를 보장합니다.

## 오늘 할 일:

- **Part A**: 문서를 청크(CHUNKS)로 나누기
- **Part B**: 청크를 벡터(VECTORS)로 인코딩하여 Chroma에 저장
- **Part C**: 벡터를 시각화하기

> 💡 **전문가 조언**: RAG의 성능은 청크 분할 전략에 크게 좌우됩니다. 청크가 너무 크면 관련 없는 내용이 섞이고, 너무 작으면 문맥이 끊깁니다. `chunk_size`와 `chunk_overlap` 튜닝이 RAG 품질 향상의 핵심입니다.

### PART A: 문서를 청크(Chunks)로 나누기

In [1]:
# 필요한 라이브러리 임포트
# tiktoken: 토큰 수 계산 | langchain: 문서 로딩/분할/임베딩/벡터 저장
# TSNE: 고차원 벡터를 2D/3D로 축소하여 시각화 | plotly: 인터랙티브 그래프
import os
import glob
import tiktoken
import numpy as np
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sklearn.manifold import TSNE
import plotly.graph_objects as go

In [2]:
# 비용이 중요한 요소이므로 저렴한 모델을 사용합니다
# vector_db: Chroma 벡터 저장소가 로컬 디스크에 저장될 디렉토리명
MODEL = "gpt-4.1-nano"
db_name = "vector_db"
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")


OpenAI API Key exists and begins sk-proj-


In [3]:
# 지식 베이스 전체의 문자 수를 파악합니다
# 💡 문서 총량을 먼저 파악하면 적절한 청크 크기와 임베딩 비용을 예측할 수 있습니다
knowledge_base_path = "knowledge-base/**/*.md"
files = glob.glob(knowledge_base_path, recursive=True)
print(f"Found {len(files)} files in the knowledge base")

entire_knowledge_base = ""

for file_path in files:
    with open(file_path, 'r', encoding='utf-8') as f:
        entire_knowledge_base += f.read()
        entire_knowledge_base += "\n\n"

print(f"Total characters in knowledge base: {len(entire_knowledge_base):,}")

Found 76 files in the knowledge base
Total characters in knowledge base: 304,434


In [4]:
# 모든 문서의 토큰 수를 계산합니다
# 💡 OpenAI API는 문자 수가 아닌 토큰 수로 비용이 청구됩니다
# 전체를 한 번에 컨텍스트로 넣으면 얼마나 비싼지 가늠할 수 있습니다
encoding = tiktoken.encoding_for_model(MODEL)
tokens = encoding.encode(entire_knowledge_base)
token_count = len(tokens)
print(f"Total tokens for {MODEL}: {token_count:,}")

Total tokens for gpt-4.1-nano: 63,555


In [5]:
# LangChain의 DirectoryLoader로 지식 베이스 전체를 로드합니다
# 각 폴더명(employees, products, contracts, company)을 doc_type 메타데이터로 저장
# 💡 메타데이터를 함께 저장하면 나중에 특정 문서 유형만 필터링하여 검색할 수 있습니다
folders = glob.glob("knowledge-base/*")

documents = []
for folder in folders:
    doc_type = os.path.basename(folder)
    loader = DirectoryLoader(folder, glob="**/*.md", loader_cls=TextLoader, loader_kwargs={'encoding': 'utf-8'})
    folder_docs = loader.load()
    for doc in folder_docs:
        doc.metadata["doc_type"] = doc_type
        documents.append(doc)

print(f"Loaded {len(documents)} documents")

Loaded 76 documents


In [6]:
documents[1]

Document(metadata={'source': 'knowledge-base\\company\\careers.md', 'doc_type': 'company'}, page_content="# Careers at Insurellm\n\n## Why Join Insurellm?\n\nAt Insurellm, we're not just building software—we're revolutionizing an entire industry. Since our founding in 2015, we've evolved from a high-growth startup to a lean, profitable company with 32 highly talented employees managing 32 active contracts across all eight of our product lines.\n\nAfter reaching 200 employees in 2020, we strategically restructured in 2022-2023 to focus on sustainable growth, operational excellence, and building a world-class remote-first culture. Today, we're a tight-knit team of exceptional professionals who deliver outsized impact through automation, AI, and strategic focus on high-value enterprise clients—from regional insurers to global reinsurance partners.\n\n### Our Culture\n\nWe live by our core values every day:\n- **Innovation First**: We encourage experimentation and creative problem-solving\

In [7]:
# RecursiveCharacterTextSplitter로 문서를 청크로 분할합니다
# chunk_size=1000: 청크당 최대 1000자
# chunk_overlap=200: 청크 간 200자 중복 → 문맥 연속성 유지
# 💡 overlap이 없으면 문장이 청크 경계에서 잘릴 때 의미가 손실될 수 있습니다
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)

print(f"Divided into {len(chunks)} chunks")
print(f"First chunk:\n\n{chunks[0]}")

Divided into 413 chunks
First chunk:

page_content='# About Insurellm

Insurellm was founded by Avery Lancaster in 2015 as an insurance tech startup designed to disrupt an industry in need of innovative products. Its first product was Markellm, the marketplace connecting consumers with insurance providers.

The company experienced rapid growth in its first five years, expanding its product portfolio to include Carllm (auto insurance portal), Homellm (home insurance portal), and Rellm (enterprise reinsurance platform). By 2020, Insurellm had reached a peak of 200 employees with 12 offices across the US.' metadata={'source': 'knowledge-base\\company\\about.md', 'doc_type': 'company'}


In [8]:
chunks[100]

Document(metadata={'source': 'knowledge-base\\contracts\\Contract with GlobalRe Partners for Rellm.md', 'doc_type': 'contracts'}, page_content='13. **Climate Risk Analytics:** Forward-looking climate modeling:\n    - IPCC climate scenario analysis (RCP 2.6, 4.5, 8.5)\n    - Transition risk assessment\n    - Physical risk modeling for perils (hurricane, wildfire, flood, drought)\n    - Sea level rise impact analysis\n    - Temperature trend incorporation\n    - Climate-adjusted pricing recommendations\n    - Stranded asset identification\n    - Green reinsurance opportunities\n\n---\n\n## Support\n\nInsurellm commits to comprehensive Enterprise-level support for GlobalRe Partners:\n\n1. **Dedicated Success Team:**\n   - Executive sponsor (CEO-level) with quarterly strategic reviews\n   - Dedicated Senior Vice President of Customer Success with bi-weekly engagement\n   - Technical Account Manager for platform optimization\n   - Solutions Architect team (2 FTE) for strategic initiatives\n

### PART B: 벡터를 만들어 Chroma에 저장하기

3주차에서 Hugging Face 계정을 만들고 `HF_TOKEN`을 발급받았습니다.

이 시점에서 `.env` 파일에 추가하고 `load_dotenv(override=True)`를 실행하는 것이 좋습니다.

(실제로는 필수가 아닐 수 있습니다.)

> 💡 **전문가 조언**: HuggingFace의 `all-MiniLM-L6-v2`는 무료이면서 성능이 뛰어난 임베딩 모델입니다. OpenAI 임베딩 대비 비용이 0원이므로, 비용 최적화가 중요한 프로젝트에서 먼저 시도해볼 만한 옵션입니다.

In [9]:
# 임베딩 모델 선택
# 현재: HuggingFace 무료 모델 (로컬 실행, 비용 없음)
# 대안: OpenAI text-embedding-3-large (더 높은 성능, 유료)
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
#embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

# 기존 DB가 있으면 삭제하고 새로 생성 (재실행 시 중복 방지)
if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()

# 청크를 임베딩하여 Chroma 벡터 저장소에 저장 (persist_directory로 디스크에 영구 저장)
vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_name)
print(f"Vectorstore created with {vectorstore._collection.count()} documents")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\yeop\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\yeop\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Vectorstore created with 413 documents


In [10]:
# 벡터 저장소의 내부 구조를 살펴봅니다
# 💡 all-MiniLM-L6-v2는 384차원 벡터를 생성합니다
# text-embedding-3-large는 3072차원 → 더 정밀하지만 저장 공간/비용 증가
collection = vectorstore._collection
count = collection.count()

sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
dimensions = len(sample_embedding)
print(f"There are {count:,} vectors with {dimensions:,} dimensions in the vector store")

There are 413 vectors with 384 dimensions in the vector store


### Part C: 시각화!

> 💡 **전문가 조언**: 벡터 시각화는 임베딩 모델이 문서를 얼마나 잘 구분하는지 직관적으로 확인하는 강력한 방법입니다. 같은 유형의 문서(직원/제품/계약서)가 공간적으로 클러스터를 이루면 임베딩 품질이 좋다는 신호입니다.

In [11]:
# 시각화 사전 준비: 벡터, 문서, 메타데이터 추출
# 문서 유형별로 색상을 지정하여 시각화 시 구분이 되도록 설정
# products=파란색, employees=초록색, contracts=빨간색, company=주황색
result = collection.get(include=['embeddings', 'documents', 'metadatas'])
vectors = np.array(result['embeddings'])
documents = result['documents']
metadatas = result['metadatas']
doc_types = [metadata['doc_type'] for metadata in metadatas]
colors = [['blue', 'green', 'red', 'orange'][['products', 'employees', 'contracts', 'company'].index(t)] for t in doc_types]

In [12]:
# 사람은 2D를 더 쉽게 이해합니다!
# t-SNE(t-분포 확률적 이웃 임베딩)로 고차원 벡터를 2D로 차원 축소합니다
# 💡 t-SNE는 가까운 점들의 관계를 보존하여 클러스터 구조를 잘 드러냅니다
# random_state=42: 재현 가능한 결과를 위한 고정 시드값

tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# 2D 산점도 생성: 마우스 오버 시 문서 유형과 텍스트 미리보기 표시
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(title='2D Chroma 벡터 저장소 시각화',
    scene=dict(xaxis_title='x',yaxis_title='y'),
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

In [13]:
# 3D로도 시도해 봅시다!
# 3D는 2D보다 더 많은 분리 구조를 보여줄 수 있습니다
# 💡 Plotly 3D 산점도는 인터랙티브하게 회전/확대 가능합니다

tsne = TSNE(n_components=3, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# 3D 산점도 생성
fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 2],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='3D Chroma 벡터 저장소 시각화',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    width=900,
    height=700,
    margin=dict(r=10, b=10, l=10, t=40)
)

fig.show()